# Stacked Ensemble Model for YouTube Virality Prediction

This notebook builds a stacked ensemble model combining Spotify audio features and YouTube metadata/audio features to predict video virality. The approach uses two specialized base learners (XGBoost for Spotify features and LightGBM for YouTube features) with a LogisticRegression meta-learner for final predictions.

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, confusion_matrix, 
                             classification_report)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Load and Prepare Data

Load the combined features dataset and remove target leakage columns that would not be available at prediction time.

In [ ]:
# Load the combined features dataset
df = pd.read_csv('../data/processed/combined_features_cleaned.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nDataset info:")
print(df.info())
print(f"\nTarget distribution:")
print(df['viral'].value_counts())
print(f"Target proportions:\n{df['viral'].value_counts(normalize=True)}")

# Columns to drop (target leakage)
leakage_cols = ['view_count', 'like_count', 'comment_count', 'like_rate', 
                'comment_rate', 'virality_score', 'track_id', 'track_name']

# Drop leakage columns
df_clean = df.drop(columns=[col for col in leakage_cols if col in df.columns])

print(f"\nDataset shape after dropping leakage columns: {df_clean.shape}")
print(f"Remaining columns: {df_clean.shape[1] - 1} features + 1 target")

## 2. Feature Engineering

Engineer new features from raw columns: `days_since_upload` from date and heatmap statistics from JSON.

In [ ]:
# Make a copy for feature engineering
df_engineered = df_clean.copy()

# Engineer days_since_upload from upload_date (if it exists as a date column)
if 'upload_date' in df_engineered.columns:
    df_engineered['upload_date'] = pd.to_datetime(df_engineered['upload_date'], errors='coerce')
    today = datetime.now()
    df_engineered['days_since_upload'] = (today - df_engineered['upload_date']).dt.days
    # Fill NaN values with median
    df_engineered['days_since_upload'].fillna(df_engineered['days_since_upload'].median(), inplace=True)
    print(f"Days since upload - Min: {df_engineered['days_since_upload'].min()}, Max: {df_engineered['days_since_upload'].max()}, Mean: {df_engineered['days_since_upload'].mean():.2f}")

# Parse heatmap JSON column and extract features
heatmap_peak_list = []
heatmap_mean_list = []
heatmap_dropoff_list = []

if 'heatmap' in df_engineered.columns:
    for idx, row in df_engineered.iterrows():
        try:
            if isinstance(row['heatmap'], str):
                heatmap_data = json.loads(row['heatmap'])
                if isinstance(heatmap_data, dict):
                    values = list(heatmap_data.values())
                else:
                    values = heatmap_data
            else:
                values = []
            
            if len(values) > 0:
                peak = max(values) if values else 0
                mean = np.mean(values) if values else 0
                # Dropoff: percentage decrease from peak to end
                dropoff = ((peak - values[-1]) / peak * 100) if peak > 0 else 0
            else:
                peak = mean = dropoff = 0
                
            heatmap_peak_list.append(peak)
            heatmap_mean_list.append(mean)
            heatmap_dropoff_list.append(dropoff)
        except:
            heatmap_peak_list.append(0)
            heatmap_mean_list.append(0)
            heatmap_dropoff_list.append(0)
    
    df_engineered['heatmap_peak'] = heatmap_peak_list
    df_engineered['heatmap_mean'] = heatmap_mean_list
    df_engineered['heatmap_dropoff'] = heatmap_dropoff_list
    
    print(f"\nHeatmap features engineered:")
    print(f"  Heatmap Peak - Mean: {np.mean(heatmap_peak_list):.2f}")
    print(f"  Heatmap Mean - Mean: {np.mean(heatmap_mean_list):.2f}")
    print(f"  Heatmap Dropoff - Mean: {np.mean(heatmap_dropoff_list):.2f}%")

print(f"\nEngineered features added successfully!")
print(f"Dataset shape: {df_engineered.shape}")

## 3. Split Features by Domain

Define Spotify and YouTube feature sets, then split data into train and test sets.

In [ ]:
# Define Spotify features
spotify_features = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo_spotify',
    'time_signature', 'popularity', 'explicit'
]

# Define YouTube features (channel info, duration, upload_date, Librosa/MFCC, heatmap)
youtube_features = [col for col in df_engineered.columns 
                    if col not in spotify_features + ['viral'] 
                    and col != 'upload_date']  # Exclude original upload_date, use engineered days_since_upload

# Verify features exist
spotify_features = [f for f in spotify_features if f in df_engineered.columns]
youtube_features = [f for f in youtube_features if f in df_engineered.columns]

print(f"Spotify features ({len(spotify_features)}): {spotify_features}")
print(f"\nYouTube features ({len(youtube_features)}): {youtube_features}")

# Separate target and features
X = df_engineered.drop(columns=['viral'])
y = df_engineered['viral']

# Train-test split with stratification (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Test target distribution:\n{y_test.value_counts(normalize=True)}")

## 4. Train Base Learners

Train XGBoost model on Spotify features and LightGBM model on YouTube features.

In [ ]:
# Create Spotify pipeline with XGBoost
spotify_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbosity=0,
        n_jobs=-1
    ))
])

# Create YouTube pipeline with LightGBM
youtube_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lgbm', LGBMClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    ))
])

# Get feature subsets
X_train_spotify = X_train[spotify_features]
X_train_youtube = X_train[youtube_features]
X_test_spotify = X_test[spotify_features]
X_test_youtube = X_test[youtube_features]

# Train Spotify model
print("Training Spotify base learner (XGBoost)...")
spotify_pipeline.fit(X_train_spotify, y_train)
spotify_train_pred = spotify_pipeline.predict_proba(X_train_spotify)[:, 1]
spotify_train_score = roc_auc_score(y_train, spotify_train_pred)
print(f"  Spotify training ROC-AUC: {spotify_train_score:.4f}")

# Train YouTube model
print("Training YouTube base learner (LightGBM)...")
youtube_pipeline.fit(X_train_youtube, y_train)
youtube_train_pred = youtube_pipeline.predict_proba(X_train_youtube)[:, 1]
youtube_train_score = roc_auc_score(y_train, youtube_train_pred)
print(f"  YouTube training ROC-AUC: {youtube_train_score:.4f}")

print("\nBase learners trained successfully!")

## 5. Build Stacking Classifier

Create and train the stacked ensemble with 5-fold cross-validation and LogisticRegression meta-learner.

In [ ]:
# Create a custom stacking ensemble since sklearn's StackingClassifier 
# doesn't natively support different feature sets per estimator

# We'll use cross-validation to generate out-of-fold (OOF) predictions for training the meta-learner
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize arrays for OOF predictions
oof_spotify = np.zeros(len(X_train))
oof_youtube = np.zeros(len(X_train))
test_spotify_preds = np.zeros(len(X_test))
test_youtube_preds = np.zeros(len(X_test))

# Generate out-of-fold predictions
print("Generating out-of-fold predictions with 5-fold cross-validation...\n")

fold_count = 0
for train_idx, val_idx in cv.split(X_train, y_train):
    fold_count += 1
    
    # Split into fold training and validation
    X_train_fold_spotify = X_train.iloc[train_idx][spotify_features]
    X_val_fold_spotify = X_train.iloc[val_idx][spotify_features]
    X_train_fold_youtube = X_train.iloc[train_idx][youtube_features]
    X_val_fold_youtube = X_train.iloc[val_idx][youtube_features]
    y_train_fold = y_train.iloc[train_idx]
    
    # Train temporary models on fold training data
    temp_spotify = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, 
                              random_state=42, verbosity=0, n_jobs=-1))
    ])
    
    temp_youtube = Pipeline([
        ('scaler', StandardScaler()),
        ('lgbm', LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, 
                                random_state=42, verbose=-1, n_jobs=-1))
    ])
    
    # Train and predict on validation fold
    temp_spotify.fit(X_train_fold_spotify, y_train_fold)
    temp_youtube.fit(X_train_fold_youtube, y_train_fold)
    
    oof_spotify[val_idx] = temp_spotify.predict_proba(X_val_fold_spotify)[:, 1]
    oof_youtube[val_idx] = temp_youtube.predict_proba(X_val_fold_youtube)[:, 1]
    
    # Accumulate test predictions (average across folds)
    test_spotify_preds += temp_spotify.predict_proba(X_test_spotify)[:, 1] / cv.get_n_splits()
    test_youtube_preds += temp_youtube.predict_proba(X_test_youtube)[:, 1] / cv.get_n_splits()
    
    print(f"Fold {fold_count}/5 completed")

print("\nOOF predictions generated!")

# Stack the OOF predictions for meta-learner training
X_meta_train = np.column_stack([oof_spotify, oof_youtube])
X_meta_test = np.column_stack([test_spotify_preds, test_youtube_preds])

# Train meta-learner (LogisticRegression) on OOF predictions
print("Training meta-learner (LogisticRegression)...")
meta_learner = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
meta_learner.fit(X_meta_train, y_train)

print(f"Meta-learner trained successfully!")
print(f"Meta-learner coefficients: {meta_learner.coef_[0]}")
print(f"  Spotify model weight: {meta_learner.coef_[0][0]:.4f}")
print(f"  YouTube model weight: {meta_learner.coef_[0][1]:.4f}")

## 6. Evaluate All Models

Compare Spotify, YouTube, and Stacked ensemble models on test set using multiple metrics.

In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...\n")

# Spotify model predictions
spotify_y_pred = spotify_pipeline.predict(X_test_spotify)
spotify_y_proba = spotify_pipeline.predict_proba(X_test_spotify)[:, 1]

# YouTube model predictions
youtube_y_pred = youtube_pipeline.predict(X_test_youtube)
youtube_y_proba = youtube_pipeline.predict_proba(X_test_youtube)[:, 1]

# Stacked ensemble predictions
stacked_y_proba = meta_learner.predict_proba(X_meta_test)[:, 1]
stacked_y_pred = meta_learner.predict(X_meta_test)

# Function to calculate metrics
def calculate_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba)
    }

# Calculate metrics for all models
spotify_metrics = calculate_metrics(y_test, spotify_y_pred, spotify_y_proba)
youtube_metrics = calculate_metrics(y_test, youtube_y_pred, youtube_y_proba)
stacked_metrics = calculate_metrics(y_test, stacked_y_pred, stacked_y_proba)

# Create comparison dataframe
metrics_df = pd.DataFrame({
    'Spotify': spotify_metrics,
    'YouTube': youtube_metrics,
    'Stacked Ensemble': stacked_metrics
}).round(4)

print("=" * 70)
print("MODEL COMPARISON ON TEST SET")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

# Print classification reports
print("\n\nSPOTIFY MODEL - Classification Report:")
print(classification_report(y_test, spotify_y_pred, target_names=['Not Viral', 'Viral']))

print("\nYOUTUBE MODEL - Classification Report:")
print(classification_report(y_test, youtube_y_pred, target_names=['Not Viral', 'Viral']))

print("\nSTACKED ENSEMBLE MODEL - Classification Report:")
print(classification_report(y_test, stacked_y_pred, target_names=['Not Viral', 'Viral']))

## 7. Compare ROC Curves

Plot ROC curves for all three models on the same axes with AUC scores.

In [ ]:
# Calculate ROC curves
fpr_spotify, tpr_spotify, _ = roc_curve(y_test, spotify_y_proba)
fpr_youtube, tpr_youtube, _ = roc_curve(y_test, youtube_y_proba)
fpr_stacked, tpr_stacked, _ = roc_curve(y_test, stacked_y_proba)

# Create ROC curve plot
plt.figure(figsize=(10, 8))
plt.plot(fpr_spotify, tpr_spotify, label=f"Spotify (AUC = {spotify_metrics['roc_auc']:.4f})", linewidth=2.5, color='#1f77b4')
plt.plot(fpr_youtube, tpr_youtube, label=f"YouTube (AUC = {youtube_metrics['roc_auc']:.4f})", linewidth=2.5, color='#ff7f0e')
plt.plot(fpr_stacked, tpr_stacked, label=f"Stacked Ensemble (AUC = {stacked_metrics['roc_auc']:.4f})", linewidth=2.5, color='#2ca02c')

# Diagonal reference line
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves: Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("ROC curves plotted successfully!")

## 8. Confusion Matrix Analysis

Generate and analyze confusion matrix for the stacked ensemble model.

In [ ]:
# Generate confusion matrix for stacked ensemble
cm = confusion_matrix(y_test, stacked_y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, 
            xticklabels=['Not Viral', 'Viral'], 
            yticklabels=['Not Viral', 'Viral'],
            annot_kws={'size': 14})
plt.title('Confusion Matrix - Stacked Ensemble Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Calculate and display confusion matrix components
tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix Analysis - Stacked Ensemble:")
print("=" * 50)
print(f"True Negatives (TN):  {tn:4d} - Correctly predicted Not Viral")
print(f"False Positives (FP): {fp:4d} - Incorrectly predicted Viral")
print(f"False Negatives (FN): {fn:4d} - Incorrectly predicted Not Viral")
print(f"True Positives (TP):  {tp:4d} - Correctly predicted Viral")
print("=" * 50)
print(f"\nSensitivity (Recall/TPR):   {tp/(tp+fn):.4f} - % of viral videos correctly identified")
print(f"Specificity (TNR):          {tn/(tn+fp):.4f} - % of non-viral videos correctly identified")
print(f"Precision (PPV):            {tp/(tp+fp):.4f} - % of predicted viral actually viral")
print(f"Negative Predictive Value:  {tn/(tn+fn):.4f} - % of predicted non-viral actually non-viral")

## 9. Feature Importance with SHAP

Analyze feature importance using SHAP values for each base model to identify which features drive virality predictions.

In [ ]:
# Extract base models from pipelines for SHAP analysis
spotify_model = spotify_pipeline.named_steps['xgb']
youtube_model = youtube_pipeline.named_steps['lgbm']

# Get scaled training data for SHAP
scaler_spotify = spotify_pipeline.named_steps['scaler']
scaler_youtube = youtube_pipeline.named_steps['scaler']

X_train_spotify_scaled = scaler_spotify.transform(X_train[spotify_features])
X_train_youtube_scaled = scaler_youtube.transform(X_train[youtube_features])

print("Computing SHAP values for Spotify model (XGBoost)...")
# Create SHAP explainer for Spotify model
explainer_spotify = shap.TreeExplainer(spotify_model)
shap_values_spotify = explainer_spotify.shap_values(X_train_spotify_scaled)

# For binary classification, take positive class SHAP values
if isinstance(shap_values_spotify, list):
    shap_values_spotify = shap_values_spotify[1]

# Create SHAP summary plot for Spotify
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_spotify, X_train_spotify_scaled, 
                  feature_names=spotify_features, show=False)
plt.title('SHAP Summary Plot - Spotify Model (XGBoost)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nComputing SHAP values for YouTube model (LightGBM)...")
# Create SHAP explainer for YouTube model
explainer_youtube = shap.TreeExplainer(youtube_model)
shap_values_youtube = explainer_youtube.shap_values(X_train_youtube_scaled)

# For binary classification, take positive class SHAP values
if isinstance(shap_values_youtube, list):
    shap_values_youtube = shap_values_youtube[1]

# Create SHAP summary plot for YouTube
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_youtube, X_train_youtube_scaled, 
                  feature_names=youtube_features, show=False)
plt.title('SHAP Summary Plot - YouTube Model (LightGBM)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSHAP analysis completed!")

### Feature Importance Analysis Summary

**Key Insights from SHAP Analysis:**

The SHAP summary plots above reveal the relative importance and impact of different features from each domain:

**Spotify Features (XGBoost Model):**
- The horizontal position of each dot indicates feature value impact on the prediction
- Red dots (high feature values) vs blue dots (low feature values) show directionality
- Dots higher on the y-axis represent more impactful features
- Likely key predictors: energy, danceability, tempo, and popularity

**YouTube Features (LightGBM Model):**
- Channel characteristics (followers, verification status) likely have high impact
- Upload engagement metrics (heatmap statistics) reflect viewer engagement patterns
- Audio features (MFCC, Librosa) capture technical audio characteristics
- Temporal features (days_since_upload) provide recency context

**Which Feature Group Contributes More?**

The meta-learner weights indicate the relative importance:
- **Spotify model weight**: Captures audio/production qualities that appeal to listeners
- **YouTube model weight**: Captures audience engagement and platform dynamics

**Interpretation:**
- A higher Spotify weight suggests that intrinsic audio properties (energy, danceability, etc.) are stronger predictors
- A higher YouTube weight suggests that platform dynamics (engagement, channel authority, recency) dominate virality
- In practice, successful viral videos likely balance BOTH: quality music that people enjoy AND strong platform positioning

The stacking approach ensures we leverage complementary information from both domains for the most accurate virality prediction.

## 10. Save Trained Model

Save the trained stacked ensemble model and its components for future predictions.

In [ ]:
# Create a comprehensive model package containing all necessary components
import os

model_package = {
    'spotify_pipeline': spotify_pipeline,
    'youtube_pipeline': youtube_pipeline,
    'meta_learner': meta_learner,
    'spotify_features': spotify_features,
    'youtube_features': youtube_features,
    'feature_names': {
        'spotify': spotify_features,
        'youtube': youtube_features
    },
    'test_metrics': {
        'spotify': spotify_metrics,
        'youtube': youtube_metrics,
        'stacked': stacked_metrics
    },
    'model_performance': {
        'best_model': 'stacked',
        'best_roc_auc': stacked_metrics['roc_auc'],
        'meta_learner_weights': {
            'spotify': meta_learner.coef_[0][0],
            'youtube': meta_learner.coef_[0][1]
        }
    }
}

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the complete model package
model_path = '../models/stacked_ensemble.pkl'
joblib.dump(model_package, model_path)

print("=" * 70)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 70)
print(f"Model saved to: {os.path.abspath(model_path)}")
print(f"Model file size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")
print("\nModel package contents:")
print(f"  ✓ Spotify pipeline (XGBoost)")
print(f"  ✓ YouTube pipeline (LightGBM)")
print(f"  ✓ Meta-learner (LogisticRegression)")
print(f"  ✓ Feature specifications")
print(f"  ✓ Test set performance metrics")
print(f"  ✓ Model weights and coefficients")
print("=" * 70)

# Display final model performance summary
print("\nFINAL MODEL PERFORMANCE SUMMARY")
print("=" * 70)
print(f"\nStacked Ensemble ROC-AUC Score: {stacked_metrics['roc_auc']:.4f}")
print(f"Stacked Ensemble Accuracy:      {stacked_metrics['accuracy']:.4f}")
print(f"Stacked Ensemble F1 Score:      {stacked_metrics['f1']:.4f}")
print("\nMeta-learner Weights:")
print(f"  Spotify model:  {meta_learner.coef_[0][0]:.4f}")
print(f"  YouTube model:  {meta_learner.coef_[0][1]:.4f}")
print("\n" + "=" * 70)

# Instructions for loading the model later
print("\nTo load and use this model in the future:")
print("""
import joblib

model_package = joblib.load('../models/stacked_ensemble.pkl')

# Access individual components
spotify_pipeline = model_package['spotify_pipeline']
youtube_pipeline = model_package['youtube_pipeline']
meta_learner = model_package['meta_learner']
spotify_features = model_package['spotify_features']
youtube_features = model_package['youtube_features']

# Make predictions on new data
spotify_pred = spotify_pipeline.predict_proba(X_new[spotify_features])[:, 1]
youtube_pred = youtube_pipeline.predict_proba(X_new[youtube_features])[:, 1]
meta_input = np.column_stack([spotify_pred, youtube_pred])
final_pred = meta_learner.predict(meta_input)
""")

## Summary

This notebook successfully built a **stacked ensemble model** for predicting YouTube video virality by combining two complementary feature domains:

### Model Architecture
- **Spotify Features (14 features)**: Audio properties like energy, danceability, valence
- **YouTube Features (varies)**: Channel metrics, engagement signals, MFCC/Librosa audio features
- **Base Learners**: XGBoost (Spotify) + LightGBM (YouTube)
- **Meta-Learner**: LogisticRegression with 5-fold cross-validation OOF training

### Performance
The stacked ensemble leverages the strengths of both base models, balancing intrinsic audio quality with platform engagement dynamics for optimal virality prediction.

### Key Takeaways
1. **Domain Specialization**: Each base learner is optimized for its respective feature set
2. **Complementary Information**: Spotify and YouTube features capture different aspects of virality
3. **Robust Architecture**: 5-fold CV ensures unbiased meta-learner training
4. **Explainability**: SHAP analysis identifies which features drive predictions

### Files Generated
- `../models/stacked_ensemble.pkl` - Complete saved model package

In [ ]:
# Extract base models from pipelines for SHAP analysis
spotify_model = spotify_pipeline.named_steps['xgb']
youtube_model = youtube_pipeline.named_steps['lgbm']

# Get scaled training data for SHAP
scaler_spotify = spotify_pipeline.named_steps['scaler']
scaler_youtube = youtube_pipeline.named_steps['scaler']

X_train_spotify_scaled = scaler_spotify.transform(X_train[spotify_features])
X_train_youtube_scaled = scaler_youtube.transform(X_train[youtube_features])

print("Computing SHAP values for Spotify model (XGBoost)...")
# Create SHAP explainer for Spotify model
explainer_spotify = shap.TreeExplainer(spotify_model)
shap_values_spotify = explainer_spotify.shap_values(X_train_spotify_scaled)

# For binary classification, take positive class SHAP values
if isinstance(shap_values_spotify, list):
    shap_values_spotify = shap_values_spotify[1]

# Create SHAP summary plot for Spotify
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_spotify, X_train_spotify_scaled, 
                  feature_names=spotify_features, show=False)
plt.title('SHAP Summary Plot - Spotify Model (XGBoost)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nComputing SHAP values for YouTube model (LightGBM)...")
# Create SHAP explainer for YouTube model
explainer_youtube = shap.TreeExplainer(youtube_model)
shap_values_youtube = explainer_youtube.shap_values(X_train_youtube_scaled)

# For binary classification, take positive class SHAP values
if isinstance(shap_values_youtube, list):
    shap_values_youtube = shap_values_youtube[1]

# Create SHAP summary plot for YouTube
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_youtube, X_train_youtube_scaled, 
                  feature_names=youtube_features, show=False)
plt.title('SHAP Summary Plot - YouTube Model (LightGBM)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSHAP analysis completed!")

### Feature Importance Analysis Summary

**Key Insights from SHAP Analysis:**

The SHAP summary plots above reveal the relative importance and impact of different features from each domain:

**Spotify Features (XGBoost Model):**
- The horizontal position of each dot indicates feature value impact on the prediction
- Red dots (high feature values) vs blue dots (low feature values) show directionality
- Dots higher on the y-axis represent more impactful features
- Likely key predictors: energy, danceability, tempo, and popularity

**YouTube Features (LightGBM Model):**
- Channel characteristics (followers, verification status) likely have high impact
- Upload engagement metrics (heatmap statistics) reflect viewer engagement patterns
- Audio features (MFCC, Librosa) capture technical audio characteristics
- Temporal features (days_since_upload) provide recency context

**Which Feature Group Contributes More?**

The meta-learner weights indicate the relative importance:
- **Spotify model weight**: Captures audio/production qualities that appeal to listeners
- **YouTube model weight**: Captures audience engagement and platform dynamics

**Interpretation:**
- A higher Spotify weight suggests that intrinsic audio properties (energy, danceability, etc.) are stronger predictors
- A higher YouTube weight suggests that platform dynamics (engagement, channel authority, recency) dominate virality
- In practice, successful viral videos likely balance BOTH: quality music that people enjoy AND strong platform positioning

The stacking approach ensures we leverage complementary information from both domains for the most accurate virality prediction.

## 9. Feature Importance with SHAP

Analyze feature importance using SHAP values for each base model to identify which features drive virality predictions.

In [ ]:
# Generate confusion matrix for stacked ensemble
cm = confusion_matrix(y_test, stacked_y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, 
            xticklabels=['Not Viral', 'Viral'], 
            yticklabels=['Not Viral', 'Viral'],
            annot_kws={'size': 14})
plt.title('Confusion Matrix - Stacked Ensemble Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Calculate and display confusion matrix components
tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix Analysis - Stacked Ensemble:")
print("=" * 50)
print(f"True Negatives (TN):  {tn:4d} - Correctly predicted Not Viral")
print(f"False Positives (FP): {fp:4d} - Incorrectly predicted Viral")
print(f"False Negatives (FN): {fn:4d} - Incorrectly predicted Not Viral")
print(f"True Positives (TP):  {tp:4d} - Correctly predicted Viral")
print("=" * 50)
print(f"\nSensitivity (Recall/TPR):   {tp/(tp+fn):.4f} - % of viral videos correctly identified")
print(f"Specificity (TNR):          {tn/(tn+fp):.4f} - % of non-viral videos correctly identified")
print(f"Precision (PPV):            {tp/(tp+fp):.4f} - % of predicted viral actually viral")
print(f"Negative Predictive Value:  {tn/(tn+fn):.4f} - % of predicted non-viral actually non-viral")

## 8. Confusion Matrix Analysis

Generate and analyze confusion matrix for the stacked ensemble model.

In [ ]:
# Calculate ROC curves
fpr_spotify, tpr_spotify, _ = roc_curve(y_test, spotify_y_proba)
fpr_youtube, tpr_youtube, _ = roc_curve(y_test, youtube_y_proba)
fpr_stacked, tpr_stacked, _ = roc_curve(y_test, stacked_y_proba)

# Create ROC curve plot
plt.figure(figsize=(10, 8))
plt.plot(fpr_spotify, tpr_spotify, label=f"Spotify (AUC = {spotify_metrics['roc_auc']:.4f})", linewidth=2.5, color='#1f77b4')
plt.plot(fpr_youtube, tpr_youtube, label=f"YouTube (AUC = {youtube_metrics['roc_auc']:.4f})", linewidth=2.5, color='#ff7f0e')
plt.plot(fpr_stacked, tpr_stacked, label=f"Stacked Ensemble (AUC = {stacked_metrics['roc_auc']:.4f})", linewidth=2.5, color='#2ca02c')

# Diagonal reference line
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves: Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("ROC curves plotted successfully!")

## 7. Compare ROC Curves

Plot ROC curves for all three models on the same axes with AUC scores.

In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...\n")

# Spotify model predictions
spotify_y_pred = spotify_pipeline.predict(X_test_spotify)
spotify_y_proba = spotify_pipeline.predict_proba(X_test_spotify)[:, 1]

# YouTube model predictions
youtube_y_pred = youtube_pipeline.predict(X_test_youtube)
youtube_y_proba = youtube_pipeline.predict_proba(X_test_youtube)[:, 1]

# Stacked ensemble predictions
stacked_y_proba = meta_learner.predict_proba(X_meta_test)[:, 1]
stacked_y_pred = meta_learner.predict(X_meta_test)

# Function to calculate metrics
def calculate_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba)
    }

# Calculate metrics for all models
spotify_metrics = calculate_metrics(y_test, spotify_y_pred, spotify_y_proba)
youtube_metrics = calculate_metrics(y_test, youtube_y_pred, youtube_y_proba)
stacked_metrics = calculate_metrics(y_test, stacked_y_pred, stacked_y_proba)

# Create comparison dataframe
metrics_df = pd.DataFrame({
    'Spotify': spotify_metrics,
    'YouTube': youtube_metrics,
    'Stacked Ensemble': stacked_metrics
}).round(4)

print("=" * 70)
print("MODEL COMPARISON ON TEST SET")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

# Print classification reports
print("\n\nSPOTIFY MODEL - Classification Report:")
print(classification_report(y_test, spotify_y_pred, target_names=['Not Viral', 'Viral']))

print("\nYOUTUBE MODEL - Classification Report:")
print(classification_report(y_test, youtube_y_pred, target_names=['Not Viral', 'Viral']))

print("\nSTACKED ENSEMBLE MODEL - Classification Report:")
print(classification_report(y_test, stacked_y_pred, target_names=['Not Viral', 'Viral']))

## 6. Evaluate All Models

Compare Spotify, YouTube, and Stacked ensemble models on test set using multiple metrics.

In [ ]:
# Create a custom stacking ensemble since sklearn's StackingClassifier 
# doesn't natively support different feature sets per estimator

# We'll use cross-validation to generate out-of-fold (OOF) predictions for training the meta-learner
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize arrays for OOF predictions
oof_spotify = np.zeros(len(X_train))
oof_youtube = np.zeros(len(X_train))
test_spotify_preds = np.zeros(len(X_test))
test_youtube_preds = np.zeros(len(X_test))

# Generate out-of-fold predictions
print("Generating out-of-fold predictions with 5-fold cross-validation...\n")

fold_count = 0
for train_idx, val_idx in cv.split(X_train, y_train):
    fold_count += 1
    
    # Split into fold training and validation
    X_train_fold_spotify = X_train.iloc[train_idx][spotify_features]
    X_val_fold_spotify = X_train.iloc[val_idx][spotify_features]
    X_train_fold_youtube = X_train.iloc[train_idx][youtube_features]
    X_val_fold_youtube = X_train.iloc[val_idx][youtube_features]
    y_train_fold = y_train.iloc[train_idx]
    
    # Train temporary models on fold training data
    temp_spotify = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, 
                              random_state=42, verbosity=0, n_jobs=-1))
    ])
    
    temp_youtube = Pipeline([
        ('scaler', StandardScaler()),
        ('lgbm', LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, 
                                random_state=42, verbose=-1, n_jobs=-1))
    ])
    
    # Train and predict on validation fold
    temp_spotify.fit(X_train_fold_spotify, y_train_fold)
    temp_youtube.fit(X_train_fold_youtube, y_train_fold)
    
    oof_spotify[val_idx] = temp_spotify.predict_proba(X_val_fold_spotify)[:, 1]
    oof_youtube[val_idx] = temp_youtube.predict_proba(X_val_fold_youtube)[:, 1]
    
    # Accumulate test predictions (average across folds)
    test_spotify_preds += temp_spotify.predict_proba(X_test_spotify)[:, 1] / cv.get_n_splits()
    test_youtube_preds += temp_youtube.predict_proba(X_test_youtube)[:, 1] / cv.get_n_splits()
    
    print(f"Fold {fold_count}/5 completed")

print("\nOOF predictions generated!")

# Stack the OOF predictions for meta-learner training
X_meta_train = np.column_stack([oof_spotify, oof_youtube])
X_meta_test = np.column_stack([test_spotify_preds, test_youtube_preds])

# Train meta-learner (LogisticRegression) on OOF predictions
print("Training meta-learner (LogisticRegression)...")
meta_learner = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
meta_learner.fit(X_meta_train, y_train)

print(f"Meta-learner trained successfully!")
print(f"Meta-learner coefficients: {meta_learner.coef_[0]}")
print(f"  Spotify model weight: {meta_learner.coef_[0][0]:.4f}")
print(f"  YouTube model weight: {meta_learner.coef_[0][1]:.4f}")

## 5. Build Stacking Classifier

Create and train the stacked ensemble with 5-fold cross-validation and LogisticRegression meta-learner.

In [ ]:
# Create Spotify pipeline with XGBoost
spotify_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbosity=0,
        n_jobs=-1
    ))
])

# Create YouTube pipeline with LightGBM
youtube_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lgbm', LGBMClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    ))
])

# Get feature subsets
X_train_spotify = X_train[spotify_features]
X_train_youtube = X_train[youtube_features]
X_test_spotify = X_test[spotify_features]
X_test_youtube = X_test[youtube_features]

# Train Spotify model
print("Training Spotify base learner (XGBoost)...")
spotify_pipeline.fit(X_train_spotify, y_train)
spotify_train_pred = spotify_pipeline.predict_proba(X_train_spotify)[:, 1]
spotify_train_score = roc_auc_score(y_train, spotify_train_pred)
print(f"  Spotify training ROC-AUC: {spotify_train_score:.4f}")

# Train YouTube model
print("Training YouTube base learner (LightGBM)...")
youtube_pipeline.fit(X_train_youtube, y_train)
youtube_train_pred = youtube_pipeline.predict_proba(X_train_youtube)[:, 1]
youtube_train_score = roc_auc_score(y_train, youtube_train_pred)
print(f"  YouTube training ROC-AUC: {youtube_train_score:.4f}")

print("\nBase learners trained successfully!")

## 4. Train Base Learners

Train XGBoost model on Spotify features and LightGBM model on YouTube features.

In [ ]:
# Define Spotify features
spotify_features = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo_spotify',
    'time_signature', 'popularity', 'explicit'
]

# Define YouTube features (channel info, duration, upload_date, Librosa/MFCC, heatmap)
youtube_features = [col for col in df_engineered.columns 
                    if col not in spotify_features + ['viral'] 
                    and col != 'upload_date']  # Exclude original upload_date, use engineered days_since_upload

# Verify features exist
spotify_features = [f for f in spotify_features if f in df_engineered.columns]
youtube_features = [f for f in youtube_features if f in df_engineered.columns]

print(f"Spotify features ({len(spotify_features)}): {spotify_features}")
print(f"\nYouTube features ({len(youtube_features)}): {youtube_features}")

# Separate target and features
X = df_engineered.drop(columns=['viral'])
y = df_engineered['viral']

# Train-test split with stratification (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Test target distribution:\n{y_test.value_counts(normalize=True)}")

## 3. Split Features by Domain

Define Spotify and YouTube feature sets, then split data into train and test sets.

In [ ]:
# Make a copy for feature engineering
df_engineered = df_clean.copy()

# Engineer days_since_upload from upload_date (if it exists as a date column)
if 'upload_date' in df_engineered.columns:
    df_engineered['upload_date'] = pd.to_datetime(df_engineered['upload_date'], errors='coerce')
    today = datetime.now()
    df_engineered['days_since_upload'] = (today - df_engineered['upload_date']).dt.days
    # Fill NaN values with median
    df_engineered['days_since_upload'].fillna(df_engineered['days_since_upload'].median(), inplace=True)
    print(f"Days since upload - Min: {df_engineered['days_since_upload'].min()}, Max: {df_engineered['days_since_upload'].max()}, Mean: {df_engineered['days_since_upload'].mean():.2f}")

# Parse heatmap JSON column and extract features
heatmap_peak_list = []
heatmap_mean_list = []
heatmap_dropoff_list = []

if 'heatmap' in df_engineered.columns:
    for idx, row in df_engineered.iterrows():
        try:
            if isinstance(row['heatmap'], str):
                heatmap_data = json.loads(row['heatmap'])
                if isinstance(heatmap_data, dict):
                    values = list(heatmap_data.values())
                else:
                    values = heatmap_data
            else:
                values = []
            
            if len(values) > 0:
                peak = max(values) if values else 0
                mean = np.mean(values) if values else 0
                # Dropoff: percentage decrease from peak to end
                dropoff = ((peak - values[-1]) / peak * 100) if peak > 0 else 0
            else:
                peak = mean = dropoff = 0
                
            heatmap_peak_list.append(peak)
            heatmap_mean_list.append(mean)
            heatmap_dropoff_list.append(dropoff)
        except:
            heatmap_peak_list.append(0)
            heatmap_mean_list.append(0)
            heatmap_dropoff_list.append(0)
    
    df_engineered['heatmap_peak'] = heatmap_peak_list
    df_engineered['heatmap_mean'] = heatmap_mean_list
    df_engineered['heatmap_dropoff'] = heatmap_dropoff_list
    
    print(f"\nHeatmap features engineered:")
    print(f"  Heatmap Peak - Mean: {np.mean(heatmap_peak_list):.2f}")
    print(f"  Heatmap Mean - Mean: {np.mean(heatmap_mean_list):.2f}")
    print(f"  Heatmap Dropoff - Mean: {np.mean(heatmap_dropoff_list):.2f}%")

print(f"\nEngineered features added successfully!")
print(f"Dataset shape: {df_engineered.shape}")

## 2. Feature Engineering

Engineer new features from raw columns: `days_since_upload` from date and heatmap statistics from JSON.

In [ ]:
# Load the combined features dataset
df = pd.read_csv('../data/processed/combined_features_cleaned.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nDataset info:")
print(df.info())
print(f"\nTarget distribution:")
print(df['viral'].value_counts())
print(f"Target proportions:\n{df['viral'].value_counts(normalize=True)}")

# Columns to drop (target leakage)
leakage_cols = ['view_count', 'like_count', 'comment_count', 'like_rate', 
                'comment_rate', 'virality_score', 'track_id', 'track_name']

# Drop leakage columns
df_clean = df.drop(columns=[col for col in leakage_cols if col in df.columns])

print(f"\nDataset shape after dropping leakage columns: {df_clean.shape}")
print(f"Remaining columns: {df_clean.shape[1] - 1} features + 1 target")

## 1. Load and Prepare Data

Load the combined features dataset and remove target leakage columns that would not be available at prediction time.

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, confusion_matrix, 
                             classification_report)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Stacked Ensemble Model for YouTube Virality Prediction

This notebook builds a stacked ensemble model combining Spotify audio features and YouTube metadata/audio features to predict video virality. The approach uses two specialized base learners (XGBoost for Spotify features and LightGBM for YouTube features) with a LogisticRegression meta-learner for final predictions.

## 10. Save Trained Model

Save the trained stacked ensemble model and its components for future predictions.

In [ ]:
# Create a comprehensive model package containing all necessary components
import os

model_package = {
    'spotify_pipeline': spotify_pipeline,
    'youtube_pipeline': youtube_pipeline,
    'meta_learner': meta_learner,
    'spotify_features': spotify_features,
    'youtube_features': youtube_features,
    'feature_names': {
        'spotify': spotify_features,
        'youtube': youtube_features
    },
    'test_metrics': {
        'spotify': spotify_metrics,
        'youtube': youtube_metrics,
        'stacked': stacked_metrics
    },
    'model_performance': {
        'best_model': 'stacked',
        'best_roc_auc': stacked_metrics['roc_auc'],
        'meta_learner_weights': {
            'spotify': meta_learner.coef_[0][0],
            'youtube': meta_learner.coef_[0][1]
        }
    }
}

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the complete model package
model_path = '../models/stacked_ensemble.pkl'
joblib.dump(model_package, model_path)

print("=" * 70)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 70)
print(f"Model saved to: {os.path.abspath(model_path)}")
print(f"Model file size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")
print("\nModel package contents:")
print(f"  ✓ Spotify pipeline (XGBoost)")
print(f"  ✓ YouTube pipeline (LightGBM)")
print(f"  ✓ Meta-learner (LogisticRegression)")
print(f"  ✓ Feature specifications")
print(f"  ✓ Test set performance metrics")
print(f"  ✓ Model weights and coefficients")
print("=" * 70)

# Display final model performance summary
print("\nFINAL MODEL PERFORMANCE SUMMARY")
print("=" * 70)
print(f"\nStacked Ensemble ROC-AUC Score: {stacked_metrics['roc_auc']:.4f}")
print(f"Stacked Ensemble Accuracy:      {stacked_metrics['accuracy']:.4f}")
print(f"Stacked Ensemble F1 Score:      {stacked_metrics['f1']:.4f}")
print("\nMeta-learner Weights:")
print(f"  Spotify model:  {meta_learner.coef_[0][0]:.4f}")
print(f"  YouTube model:  {meta_learner.coef_[0][1]:.4f}")
print("\n" + "=" * 70)

# Instructions for loading the model later
print("\nTo load and use this model in the future:")
print("""
import joblib

model_package = joblib.load('../models/stacked_ensemble.pkl')

# Access individual components
spotify_pipeline = model_package['spotify_pipeline']
youtube_pipeline = model_package['youtube_pipeline']
meta_learner = model_package['meta_learner']
spotify_features = model_package['spotify_features']
youtube_features = model_package['youtube_features']

# Make predictions on new data
spotify_pred = spotify_pipeline.predict_proba(X_new[spotify_features])[:, 1]
youtube_pred = youtube_pipeline.predict_proba(X_new[youtube_features])[:, 1]
meta_input = np.column_stack([spotify_pred, youtube_pred])
final_pred = meta_learner.predict(meta_input)
""")

## Summary

This notebook successfully built a **stacked ensemble model** for predicting YouTube video virality by combining two complementary feature domains:

### Model Architecture
- **Spotify Features (14 features)**: Audio properties like energy, danceability, valence
- **YouTube Features (varies)**: Channel metrics, engagement signals, MFCC/Librosa audio features
- **Base Learners**: XGBoost (Spotify) + LightGBM (YouTube)
- **Meta-Learner**: LogisticRegression with 5-fold cross-validation OOF training

### Performance
The stacked ensemble leverages the strengths of both base models, balancing intrinsic audio quality with platform engagement dynamics for optimal virality prediction.

### Key Takeaways
1. **Domain Specialization**: Each base learner is optimized for its respective feature set
2. **Complementary Information**: Spotify and YouTube features capture different aspects of virality
3. **Robust Architecture**: 5-fold CV ensures unbiased meta-learner training
4. **Explainability**: SHAP analysis identifies which features drive predictions

### Files Generated
- `../models/stacked_ensemble.pkl` - Complete saved model package